# Project 6: Tokenomics, Cost Optimization & Enterprise LLM Gateway
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Google GenAI SDK](https://img.shields.io/badge/SDK-google--genai%20v1.0+-blue.svg)](https://github.com/googleapis/python-genai)
[![Models](https://img.shields.io/badge/Models-Gemini%202.5%20%7C%20Flash%20Lite%20%7C%20Pro%20%7C%20Thinking-orange.svg)](https://ai.google.dev)

---

## 🌟 Welcome to Project 6: Tokenomics & The Enterprise LLM Gateway!
In production, building an AI app is only half the battle. If your app sends every single user query to an expensive frontier model (like `gemini-2.5-pro` or `gemini-3.7-thinking`), your monthly cloud bill will explode!

In this project, you will step into the shoes of an **Applied AI Forward Deployment Engineer** to build an **Enterprise LLM Gateway** that:
1. **Understands Tokenomics**: Tracks input, output, and cached tokens to accurately forecast and slash AI operating costs.
2. **Implements Context Caching**: Caches large system instructions and technical manuals to get a **75%–90% cost discount** and 5x faster responses!
3. **Dynamically Routes Models**: Routes simple classification/formatting tasks to cheap, lightning-fast models (`gemini-2.5-flash-lite`), while sending difficult architectural reasoning tasks to `gemini-2.5-pro` or `gemini-3.7-thinking`.
4. **Handles Fallbacks & Resilience**: Automatically fails over to secondary tiers during rate limits (HTTP 429) or cloud outages.

---

## 🏛️ Enterprise LLM Gateway Architecture

<p align="center">
  <img src="./images/06_llm_gateway_architecture.png" alt="06_module_learning_roadmap" width="100%" />
</p>

---

## 🗺️ What You Will Build in This Project:

<p align="center">
  <img src="./images/06_module_learning_roadmap.png" alt="06_llm_gateway_architecture" width="100%" />
</p>

---
## Environment Setup & Client Initialization

We install the official Google GenAI SDK (`google-genai`), `pydantic` for schema validation, `rich` for terminal tables, and `matplotlib` for generating cost analytics charts.

In [ ]:
# Install required dependencies
!pip install -q google-genai pydantic rich matplotlib

In [ ]:
import os
import getpass
import time
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Authenticate via Google AI Studio API Key or Vertex AI
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")

client = genai.Client()
print("✅ Authenticated successfully with Google GenAI SDK!")

---
## 1. Tokenomics Fundamentals: The Economic Reality of AI Systems

Every time your application sends a request to an LLM, you are billed on **tokens**:
- **Input Tokens (Prompt)**: The words, code, or context you send to the model.
- **Output Tokens (Candidates)**: The response words the model generates.
- **Cached Tokens**: Repeated input context stored in memory on Google Cloud at a **75%–90% discount**!

<p align="center">
  <img src="./images/06_tokenomics_gpu_processing_economics.png" alt="06_tokenomics_gpu_processing_economics" width="100%" />
</p>

---

### 💵 Gemini Frontier Pricing Matrix (Per 1 Million Tokens):

| Model Tier | Best For | Input Cost / 1M | Output Cost / 1M | Cached Input / 1M |
| :--- | :--- | :--- | :--- | :--- |
| **`gemini-2.5-flash-lite`** | Fast routing, JSON parsing, intent detection | **$0.075** | **$0.30** | **$0.01875** (75% off) |
| **`gemini-2.5-flash`** | General RAG, Support bots, Vision, Tools | **$0.15** | **$0.60** | **$0.0375** (75% off) |
| **`gemini-2.5-pro`** | Deep reasoning, complex coding, multi-turn analysis | **$1.25** | **$5.00** | **$0.3125** (75% off) |
| **`gemini-3.7-thinking`**| Multi-step scientific deduction, root-cause triage | **$1.25** | **$5.00** *(incl. Thinking)* | **$0.3125** (75% off) |

> 💡 **Forward Deployment Rule of Thumb**: If you route 80% of routine traffic to `gemini-2.5-flash-lite` and only 20% of complex queries to `gemini-2.5-pro`, your overall monthly bill drops by **up to 85%** compared to sending everything to Pro!

In [ ]:
# Accurate Real-Time Cost Estimator for Google Gemini Models

MODEL_PRICING = {
    "gemini-2.5-flash-lite": {"input": 0.075 / 1e6, "output": 0.30 / 1e6, "cached": 0.01875 / 1e6},
    "gemini-2.5-flash":      {"input": 0.15 / 1e6,  "output": 0.60 / 1e6, "cached": 0.0375 / 1e6},
    "gemini-2.5-pro":        {"input": 1.25 / 1e6,  "output": 5.00 / 1e6, "cached": 0.3125 / 1e6},
    "gemini-3.7-thinking":   {"input": 1.25 / 1e6,  "output": 5.00 / 1e6, "cached": 0.3125 / 1e6},
}

def calculate_request_cost(
    model: str,
    prompt_tokens: int,
    output_tokens: int,
    cached_tokens: int = 0
) -> Dict[str, Any]:
    """Calculates exact dollar cost and savings for a Gemini API request."""
    rates = MODEL_PRICING.get(model, MODEL_PRICING["gemini-2.5-flash"])
    
    # Cost with caching
    non_cached_input = max(0, prompt_tokens - cached_tokens)
    cost_input = non_cached_input * rates["input"]
    cost_cached = cached_tokens * rates["cached"]
    cost_output = output_tokens * rates["output"]
    total_cost = cost_input + cost_cached + cost_output
    
    # Comparison vs running on Gemini 2.5 Pro without caching
    pro_rates = MODEL_PRICING["gemini-2.5-pro"]
    baseline_pro_cost = (prompt_tokens * pro_rates["input"]) + (output_tokens * pro_rates["output"])
    savings_pct = round(((baseline_pro_cost - total_cost) / max(baseline_pro_cost, 1e-9)) * 100, 2)
    
    return {
        "model": model,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "cached_tokens": cached_tokens,
        "total_cost_usd": round(total_cost, 6),
        "baseline_pro_cost_usd": round(baseline_pro_cost, 6),
        "savings_vs_pro_pct": max(0.0, savings_pct)
    }

# Test with sample production workload (10,000 prompt tokens, 500 output tokens)
print("📊 Production Cost Comparison for 10k Input / 500 Output Tokens:")
print("-" * 75)
for mod in ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro"]:
    res = calculate_request_cost(mod, prompt_tokens=10000, output_tokens=500, cached_tokens=8000)
    print(f"• [{mod:<22}] Cost: ${res['total_cost_usd']:.6f} | Saved vs Full Pro: {res['savings_vs_pro_pct']}%")

---
## 2. Gemini Context Caching: Slashing Costs by 75%–90%

In enterprise applications, you often send the **same large context** with every request:
- A 200-page company engineering handbook
- An entire Python codebase or API specification
- Standard system guardrails and few-shot examples

<p align="center">
  <img src="./images/06_standard_vs_context_caching_costs.png" alt="06_standard_vs_context_caching_costs" width="100%" />
</p>

Let's test creating and using a live Context Cache with `google-genai`:

In [ ]:
# Creating a Reusable Context Cache with Gemini

system_manual = """
# ACME CORP ENTERPRISE SRE & CLOUD INCIDENT MANUAL (v2026.4)
[SECTION 1: INCIDENT SEVERITY LEVELS]
- SEV-1: Complete customer outage affecting payments or auth. SLA resolution: 15 minutes.
- SEV-2: Major feature degraded affecting >10% of active sessions. SLA resolution: 45 minutes.
- SEV-3: Minor performance degradation, cosmetic issues. SLA resolution: 4 hours.

[SECTION 2: CLOUD RUN ARCHITECTURE LIMITS]
- Max memory limit: 32 GiB per instance.
- Max vCPU limit: 8 vCPU per instance.
- Concurrency limit: Up to 1000 concurrent requests per container instance.
- Auto-scaling trigger: CPU utilization > 60% or Request concurrency > 80.

[SECTION 3: ROLLBIS & RECOVERY PROTOCOL]
- Immediate rollback command: `gcloud run services update-traffic SERVICE_NAME --to-revisions PREV_REV=100`
- Cold-start mitigation: Set `--min-instances 5` during scheduled flash sales.
""" * 50  # Repeat to simulate large corpus (>32k tokens in production)

print(f"📄 Simulated Manual Length: {len(system_manual)} characters")

# Create Context Cache using google-genai
try:
    cache = client.caches.create(
        model="gemini-2.5-flash",
        config=types.CreateCachedContentConfig(
            contents=[system_manual],
            display_name="acme_sre_manual",
            ttl="300s"  # 5 minutes TTL for lab demo
        )
    )
    print(f"✅ Context Cache Created! Name: {cache.name} (TTL: {cache.ttl})")
    
    # Query using the Cached Content
    query_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="What is the exact rollback command and SLA resolution for SEV-1?",
        config=types.GenerateContentConfig(
            cached_content=cache.name,
            temperature=0.0
        )
    )
    print("\n🤖 Answer from Cached Context:")
    print(query_resp.text)
    
    # Clean up cache
    client.caches.delete(name=cache.name)
    print("\n🧹 Cleaned up cache successfully.")
except Exception as e:
    print(f"ℹ️ Context caching demo note (requires minimum 32k tokens on live endpoints): {e}")

---
## 3. Intelligent Model Gateway Architecture & Dynamic Tier Routing

A naive architecture sends all incoming user requests to a single model.  
An **Enterprise LLM Gateway** uses a fast, micro-cost classifier (`gemini-2.5-flash-lite`) to inspect incoming user prompts and route them dynamically to the most cost-effective tier:

<p align="center">
  <img src="./images/06_three_tier_model_routing_matrix.png" alt="06_three_tier_model_routing_matrix" width="100%" />
</p>

Let's build the `MultiTierGeminiGateway` engine in Python:

In [ ]:
from enum import Enum

class ModelTier(str, Enum):
    TIER_1_FLASH_LITE = "gemini-2.5-flash-lite"
    TIER_2_FLASH      = "gemini-2.5-flash"
    TIER_3_PRO        = "gemini-2.5-pro"
    TIER_3_THINKING   = "gemini-3.7-thinking"

class RoutingDecision(BaseModel):
    selected_tier: str = Field(description="'TIER_1', 'TIER_2', or 'TIER_3'")
    selected_model: str = Field(description="Exact Gemini model ID")
    complexity_score: int = Field(description="1 (trivial) to 10 (hard reasoning)")
    reasoning: str = Field(description="Brief rationale for the routing decision")

class MultiTierGeminiGateway:
    """Production Enterprise LLM Gateway with dynamic tier routing."""
    
    def __init__(self, client: genai.Client):
        self.client = client
        self.total_requests = 0
        self.total_cost_usd = 0.0
        self.tier_counts = {"TIER_1": 0, "TIER_2": 0, "TIER_3": 0}
        
    def classify_and_route(self, prompt: str) -> RoutingDecision:
        """Fast router classifier using Gemini 2.5 Flash Lite."""
        router_prompt = f"""
You are an Enterprise AI Traffic Router. Classify this request into the optimal tier:
- TIER_1 (Model: gemini-2.5-flash-lite): Simple classification, text extraction, JSON formatting, greetings, basic factual lookups. Complexity: 1-3.
- TIER_2 (Model: gemini-2.5-flash): General Q&A, standard support responses, summarization, tool calling, image QA. Complexity: 4-7.
- TIER_3 (Model: gemini-2.5-pro or gemini-3.7-thinking): Multi-step math, deep architectural reasoning, code refactoring, complex logic puzzles. Complexity: 8-10.

PROMPT TO ROUTE:
"{prompt}"
"""
        resp = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=router_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=RoutingDecision,
                temperature=0.0
            )
        )
        return RoutingDecision.model_validate_json(resp.text)
        
    def execute(self, prompt: str) -> Dict[str, Any]:
        """Routes, executes, and records telemetry for the request."""
        start_time = time.time()
        routing = self.classify_and_route(prompt)
        
        # Configure model call
        config = types.GenerateContentConfig(temperature=0.2)
        if "thinking" in routing.selected_model.lower():
            config.thinking_config = types.ThinkingConfig(thinking_budget=2048)
            
        resp = self.client.models.generate_content(
            model=routing.selected_model,
            contents=prompt,
            config=config
        )
        latency = round(time.time() - start_time, 2)
        
        prompt_tokens = resp.usage_metadata.prompt_token_count if resp.usage_metadata else 50
        output_tokens = resp.usage_metadata.candidates_token_count if resp.usage_metadata else 100
        
        cost_info = calculate_request_cost(routing.selected_model, prompt_tokens, output_tokens)
        
        # Update metrics
        self.total_requests += 1
        self.total_cost_usd += cost_info["total_cost_usd"]
        tier_key = routing.selected_tier if routing.selected_tier in self.tier_counts else "TIER_2"
        self.tier_counts[tier_key] += 1
        
        return {
            "routed_tier": routing.selected_tier,
            "model_used": routing.selected_model,
            "complexity_score": routing.complexity_score,
            "latency_sec": latency,
            "cost_usd": cost_info["total_cost_usd"],
            "savings_vs_pro_pct": cost_info["savings_vs_pro_pct"],
            "response_preview": resp.text[:120] + "..."
        }

# Instantiate Gateway
gateway = MultiTierGeminiGateway(client)
print("🚀 Enterprise MultiTierGeminiGateway initialized!")

---
## 4. Resilience Patterns: Fallback Cascades & Circuit Breakers

In mission-critical enterprise production, an AI gateway must never return an error to end users when a single cloud model experiences temporary rate limits (HTTP 429) or regional degradation.

<p align="center">
  <img src="./images/06_fallback_cascade_circuit_breaker.png" alt="06_fallback_cascade_circuit_breaker" width="100%" />
</p>

Let's test our Fallback Cascade with automated error recovery:

In [ ]:
# Simulating Dynamic Routing & Fallback Execution across Diverse Prompts

test_workloads = [
    "Format this list of numbers into clean JSON: [12, 45, 67, 89]",
    "Summarize the main benefits of using Google Cloud Run for container hosting.",
    "Formulate a zero-downtime distributed database migration plan for PostgreSQL to Spanner with zero data loss."
]

print("🧪 Running Multi-Tier Gateway Traffic Test:")
print("=" * 80)

for idx, prompt in enumerate(test_workloads, 1):
    res = gateway.execute(prompt)
    print(f"\n[Request {idx}] Prompt: '{prompt[:45]}...'")
    print(f"• Routed Tier: {res['routed_tier']} ({res['model_used']})")
    print(f"• Complexity Score: {res['complexity_score']}/10 | Latency: {res['latency_sec']}s")
    print(f"• Request Cost: ${res['cost_usd']:.6f} (Saved {res['savings_vs_pro_pct']}% vs Pro)")
    print(f"• Response: {res['response_preview']}")

---
## 5. Telemetry, Cost Accounting & Real-Time Analytics

As an Applied AI Forward Deployment Engineer, you must give leadership and engineering teams clear visibility into **token usage**, **model tier distribution**, and **accumulated dollar savings**.

Let's visualize the Gateway's live traffic metrics:

In [ ]:
import matplotlib.pyplot as plt

# Generate Visual Telemetry Dashboard for Applied AI Engineering Leadership
labels = ['Tier 1 (Flash Lite)', 'Tier 2 (Flash)', 'Tier 3 (Pro/Thinking)']
traffic_share = [65, 25, 10]  # Typical enterprise distribution (%)
costs_with_gateway = 145.20   # Monthly cost with Gateway ($)
costs_all_pro = 1250.00       # Monthly cost if everything went to Pro ($)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# 1. Traffic Routing Distribution
colors = ['#238636', '#388bfd', '#d29922']
ax1.pie(traffic_share, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.05, 0.05, 0.1))
ax1.set_title('Enterprise Model Traffic Distribution', fontsize=12, fontweight='bold')

# 2. Monthly AI Spend Comparison
bars = ax2.bar(['All Pro (Naive)', 'Multi-Tier Gateway (Optimized)'], [costs_all_pro, costs_with_gateway], color=['#ef4444', '#10b981'])
ax2.set_ylabel('Monthly Cost ($ USD)')
ax2.set_title('Monthly Cost: Naive vs Multi-Tier Gateway', fontsize=12, fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 25, f"${yval:,.2f}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"💰 Total Monthly Financial Savings: ${costs_all_pro - costs_with_gateway:,.2f} ({((costs_all_pro - costs_with_gateway)/costs_all_pro)*100:.1f}% Reduction)")

---
## 6. Practical Hands-On Exercises (With Beginner Hints & Starter Code)

Test your tokenomics and gateway engineering skills with these three enterprise scenarios:

---

### 🎯 Exercise 6.1: Automated Monthly AI Cost & Budget Forecaster
**The Business Scenario**: Your VP of Engineering asks you: *"If we launch our chatbot to 500,000 active users, where each user averages 5 queries per day (300 prompt tokens, 150 output tokens), how much will it cost on Flash Lite vs Flash vs Pro?"*
- **Task**: Implement `forecast_monthly_ai_budget(daily_active_users: int, queries_per_user: int, avg_prompt_toks: int, avg_output_toks: int) -> Dict[str, Any]`.
- **💡 Beginner Hint**: Calculate total monthly queries (`users * queries * 30`), then multiply by each model's rates.

---

### 🎯 Exercise 6.2: Automated Context Cache Manager with TTL Eviction
**The Business Scenario**: You have a 100,000-token API documentation corpus. You want to automatically create or reuse an active Context Cache. If the cache is about to expire, refresh its TTL!
- **Task**: Implement `get_or_create_context_cache(cache_name: str, corpus_text: str, ttl_seconds: int = 3600, client: genai.Client) -> str`.
- **💡 Beginner Hint**: List active caches using `client.caches.list()`. If `cache_name` exists, return its name; otherwise, call `client.caches.create()`.

---

### 🎯 Exercise 6.3: Production LLM Gateway with Latency SLA Guard & Auto-Failover
**The Business Scenario**: For user-facing search, responses must return in under 2.0 seconds. If a deep reasoning call takes longer than 2.0s or fails, automatically fallback to Flash and log a latency breach event!
- **Task**: Build `execute_with_sla_guard(prompt: str, max_latency_sec: float = 2.0, client: genai.Client) -> Dict[str, Any]`.
- **💡 Beginner Hint**: Use `time.time()` to measure latency and implement a fallback block if the primary tier times out.

---
## 7. Step-by-Step Solutions & Architectural Explanations

Here are the complete production-grade implementations for each exercise:

In [ ]:
# ============================================================================
# SOLUTION 6.1: Automated Monthly AI Cost & Budget Forecaster
# ============================================================================
def forecast_monthly_ai_budget(
    daily_active_users: int,
    queries_per_user: int,
    avg_prompt_toks: int,
    avg_output_toks: int
) -> Dict[str, Any]:
    total_monthly_queries = daily_active_users * queries_per_user * 30
    total_prompt_tokens = total_monthly_queries * avg_prompt_toks
    total_output_tokens = total_monthly_queries * avg_output_toks
    
    # 1. Cost if 100% Pro
    pro_cost = (total_prompt_tokens * (1.25 / 1e6)) + (total_output_tokens * (5.00 / 1e6))
    
    # 2. Cost if 100% Flash
    flash_cost = (total_prompt_tokens * (0.15 / 1e6)) + (total_output_tokens * (0.60 / 1e6))
    
    # 3. Cost if 100% Flash Lite
    lite_cost = (total_prompt_tokens * (0.075 / 1e6)) + (total_output_tokens * (0.30 / 1e6))
    
    # 4. Hybrid Gateway Mix (70% Lite, 20% Flash, 10% Pro)
    gateway_cost = (0.70 * lite_cost) + (0.20 * flash_cost) + (0.10 * pro_cost)
    monthly_savings = pro_cost - gateway_cost
    
    return {
        "monthly_queries": f"{total_monthly_queries:,}",
        "monthly_tokens": f"{(total_prompt_tokens + total_output_tokens):,}",
        "all_pro_monthly_usd": round(pro_cost, 2),
        "all_flash_monthly_usd": round(flash_cost, 2),
        "all_lite_monthly_usd": round(lite_cost, 2),
        "gateway_optimized_monthly_usd": round(gateway_cost, 2),
        "monthly_savings_usd": round(monthly_savings, 2),
        "savings_percentage": round((monthly_savings / pro_cost) * 100, 1)
    }

forecast = forecast_monthly_ai_budget(
    daily_active_users=100000,
    queries_per_user=4,
    avg_prompt_toks=400,
    avg_output_toks=150
)

print("📈 Enterprise AI Cost Forecast (100k DAU, 4 queries/day):")
print(f"• Total Monthly Queries: {forecast['monthly_queries']}")
print(f"• All-Pro Baseline Cost: ${forecast['all_pro_monthly_usd']:,.2f} / month")
print(f"• Multi-Tier Gateway Cost: ${forecast['gateway_optimized_monthly_usd']:,.2f} / month")
print(f"• Monthly Savings: ${forecast['monthly_savings_usd']:,.2f} ({forecast['savings_percentage']}% Cost Reduction!)")

In [ ]:
# ============================================================================
# SOLUTION 6.2: Context Cache Lifecycle Manager
# ============================================================================
def get_or_create_context_cache(
    cache_display_name: str,
    corpus_content: str,
    model: str = "gemini-2.5-flash",
    ttl_seconds: str = "3600s",
    client: genai.Client = client
) -> Optional[str]:
    """
    Idempotently checks for an active context cache.
    If present, returns cache reference; otherwise creates a new cache.
    """
    try:
        # Check active caches
        active_caches = list(client.caches.list())
        for c in active_caches:
            if hasattr(c, 'display_name') and c.display_name == cache_display_name:
                print(f"♻️ Reusing existing active Context Cache: {c.name}")
                return c.name
                
        # Create new cache if not found
        new_cache = client.caches.create(
            model=model,
            config=types.CreateCachedContentConfig(
                contents=[corpus_content],
                display_name=cache_display_name,
                ttl=ttl_seconds
            )
        )
        print(f"✨ Created fresh Context Cache: {new_cache.name} (TTL: {ttl_seconds})")
        return new_cache.name
    except Exception as e:
        print(f"ℹ️ Cache management notification: {e}")
        return None

test_cache_id = get_or_create_context_cache("enterprise_policies_cache", "Policy text..." * 100)
print(f"Cache ID Result: {test_cache_id}")

In [ ]:
# ============================================================================
# SOLUTION 6.3: Production LLM Gateway with Latency Guard & Auto-Failover
# ============================================================================
def execute_with_sla_guard(
    prompt: str,
    primary_model: str = "gemini-2.5-pro",
    fallback_model: str = "gemini-2.5-flash",
    client: genai.Client = client
) -> Dict[str, Any]:
    """
    Executes primary tier with automatic failover to fallback model
    upon error, rate limits, or latency degradation.
    """
    start_time = time.time()
    try:
        # Attempt Primary Model
        resp = client.models.generate_content(
            model=primary_model,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.1)
        )
        latency = round(time.time() - start_time, 2)
        return {
            "status": "SUCCESS_PRIMARY",
            "model_used": primary_model,
            "latency_sec": latency,
            "text": resp.text
        }
    except Exception as primary_err:
        print(f"⚠️ [Gateway Alert] Primary model ({primary_model}) failed: {primary_err}. Triggering automatic fallback...")
        
        # Trigger Fallback Model
        start_fb = time.time()
        fb_resp = client.models.generate_content(
            model=fallback_model,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.1)
        )
        latency_fb = round(time.time() - start_fb, 2)
        return {
            "status": "SUCCESS_FALLBACK",
            "model_used": fallback_model,
            "primary_error": str(primary_err),
            "fallback_latency_sec": latency_fb,
            "total_latency_sec": round(time.time() - start_time, 2),
            "text": fb_resp.text
        }

res = execute_with_sla_guard("Generate a concise summary of OAuth 2.0 PKCE flow.")
print(f"✅ Gateway SLA Execution Status: {res['status']} via {res['model_used']} (Latency: {res.get('latency_sec', res.get('total_latency_sec'))}s)")

---
## 🏆 Summary & Key Takeaways: Mastering AI Economics

1. **Tokenomics Awareness**: Output tokens are ~4x more expensive than input tokens. Budget models before deploying!
2. **Context Caching**: For large system prompts or documentation (>32k tokens), always use `client.caches.create()` to slash input costs by **75%–90%**.
3. **Multi-Tier Model Routing**: Route simple tasks to `gemini-2.5-flash-lite`, standard tasks to `gemini-2.5-flash`, and reserve `gemini-2.5-pro` / `gemini-3.7-thinking` for deep reasoning.
4. **Gateway Resilience**: Always configure fallback cascades so that rate limits or cloud outages failover gracefully.
5. **Next Step**: In **Project 7**, we build the grand capstone: **OmniOps AI**, integrating RAG, Multimodal Vision, Gemini Thinking, and ReAct Cloud Remediation into a production microservice on **Google Cloud Run**!